# Day 16 Capstone Mini-Project
## Predict Customer Churn — End-to-End Tabular ML

**Business problem:** Given a customer's account and plan features as of today, predict whether they will churn (`churn = 1`) so the retention team can target save-offers.

This is an imbalanced binary-classification problem. We protect **recall on churn**, while using **ROC-AUC** as the threshold-independent model-selection metric.


## Prerequisites — Generate the Offline Dataset

The dataset is generated entirely in memory. No internet connection or downloaded file is required.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification

RNG = 42
rng = np.random.default_rng(RNG)
N = 4000

X_num, y = make_classification(
    n_samples=N,
    n_features=6,
    n_informative=5,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.76, 0.24],
    flip_y=0.13,
    class_sep=0.85,
    random_state=RNG
)

Xs = (X_num - X_num.mean(0)) / X_num.std(0)

df = pd.DataFrame()

df['tenure_months'] = np.clip(Xs[:, 4] * 12 + 34, 1, 72).round().astype(int)
df['monthly_charges'] = np.clip(Xs[:, 0] * 13 + 72, 18, 130).round(2)
df['support_tickets'] = np.clip(Xs[:, 2] * 1.7 + 3, 0, 25).round().astype(int)
df['avg_latency_ms'] = np.clip(Xs[:, 3] * 35 + 120, 20, 400).round().astype(int)
df['addons_count'] = np.clip(Xs[:, 1] * 1.3 + 4, 0, 8).round().astype(int)
df['days_since_login'] = np.clip(
    -Xs[:, 4] * 8 + 16 + rng.normal(0, 4, N),
    0,
    90
).round().astype(int)

df['total_charges'] = (
    df['monthly_charges'] * df['tenure_months']
).round(2)

def by_class(p_churn, p_stay, choices):
    out = np.empty(N, dtype=object)
    mask = y == 1
    out[mask] = rng.choice(choices, mask.sum(), p=p_churn)
    out[~mask] = rng.choice(choices, (~mask).sum(), p=p_stay)
    return out

df['contract_type'] = by_class(
    [0.75, 0.18, 0.07],
    [0.34, 0.35, 0.31],
    ['Month-to-month', 'One year', 'Two year']
)

df['payment_method'] = by_class(
    [0.55, 0.20, 0.15, 0.10],
    [0.25, 0.30, 0.25, 0.20],
    ['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card']
)

df['internet_service'] = by_class(
    [0.65, 0.28, 0.07],
    [0.40, 0.35, 0.25],
    ['Fiber optic', 'DSL', 'None']
)

df['has_dependents'] = by_class(
    [0.82, 0.18],
    [0.60, 0.40],
    ['No', 'Yes']
)

# Deliberate leakage feature
df['exit_survey_sent'] = np.where(
    y == 1,
    rng.binomial(1, 0.97, N),
    rng.binomial(1, 0.02, N)
)

df['churn'] = y

print('dataset shape:', df.shape)
print('churn rate:', round(df['churn'].mean(), 3))
df.head()

## Step 1 — Frame the Problem and Explore the Data

A missed churner can be more expensive than contacting a customer who would have stayed, so recall is especially important.

In [ ]:
print(df['churn'].value_counts(normalize=True).round(3), '\n')

num_cols = [
    'tenure_months',
    'monthly_charges',
    'support_tickets',
    'avg_latency_ms',
    'addons_count',
    'days_since_login',
    'total_charges'
]

print(df.groupby('churn')[num_cols].mean().round(1).T, '\n')

print(
    df.groupby('contract_type')['churn']
      .mean()
      .round(3)
      .sort_values(ascending=False)
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for k, label in [(0, 'stay'), (1, 'churn')]:
    ax[0].hist(
        df.loc[df.churn == k, 'tenure_months'],
        bins=25,
        alpha=0.6,
        label=label
    )

ax[0].set_title('Tenure by outcome')
ax[0].set_xlabel('tenure_months')
ax[0].legend()

df.groupby('contract_type')['churn'].mean().sort_values().plot.barh(ax=ax[1])
ax[1].set_title('Churn rate by contract')
ax[1].set_xlabel('churn rate')

plt.tight_layout()
plt.show()

## Step 2 — Train / Validation / Test Split and Leakage Detection

`exit_survey_sent` is a post-outcome feature. It is only available because the customer has already churned, so it must not be used for prediction.

In [ ]:
from sklearn.model_selection import train_test_split

leak_scores = (
    df.corr(numeric_only=True)['churn']
      .drop('churn')
      .abs()
      .sort_values(ascending=False)
)

print('Absolute correlation with churn:')
print(leak_scores.round(3), '\n')

suspects = leak_scores[leak_scores > 0.9].index.tolist()
print('LEAK suspects:', suspects)

df_clean = df.drop(columns=suspects)

X = df_clean.drop(columns=['churn'])
y = df_clean['churn']

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RNG
)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp,
    y_tmp,
    test_size=0.50,
    stratify=y_tmp,
    random_state=RNG
)

print('\nshapes:', X_train.shape, X_val.shape, X_test.shape)
print(
    'churn rate train/val/test:',
    round(y_train.mean(), 3),
    round(y_val.mean(), 3),
    round(y_test.mean(), 3)
)

In [ ]:
assert 'exit_survey_sent' not in X.columns
assert X_train.shape == (2800, 11)
assert X_val.shape == (600, 11)
assert X_test.shape == (600, 11)

print('Leakage and split checks passed.')

### Why `total_charges` stays

`total_charges = monthly_charges × tenure_months` is derived, but it is available at prediction time. Derived does not automatically mean leakage. Leakage depends on whether information from the future or outcome has entered the feature.

## Step 3 — Leakage-Free Preprocessing Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_features = X_train.select_dtypes('number').columns.tolist()
cat_features = X_train.select_dtypes('object').columns.tolist()

print('numeric    :', num_features)
print('categorical:', cat_features)

pre = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

Z = pre.fit_transform(X_train)

print('\ntransformed matrix:', Z.shape)
print('feature count after one-hot:', len(pre.get_feature_names_out()))

In [ ]:
assert len(num_features) == 7
assert len(cat_features) == 4
assert Z.shape == (2800, 19)

print('Preprocessing checkpoint passed.')

## Step 4 — Compare Four Model Families with 5-Fold CV

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

models = {
    'LogReg': LogisticRegression(
        max_iter=2000,
        class_weight='balanced'
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=300,
        random_state=RNG,
        class_weight='balanced'
    ),
    'GradBoost': GradientBoostingClassifier(
        random_state=RNG
    ),
    'SVM_RBF': SVC(
        kernel='rbf',
        probability=True,
        class_weight='balanced',
        random_state=RNG
    )
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RNG
)

results = {}

for name, classifier in models.items():
    pipe = Pipeline([
        ('pre', pre),
        ('clf', classifier)
    ])

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring='roc_auc'
    )

    results[name] = scores

    print(
        f'{name:13s} ROC-AUC = '
        f'{scores.mean():.3f} +/- {scores.std():.3f}'
    )

### Model choice

The model families should score close together. We carry `GradientBoostingClassifier` into tuning because it is a strong tabular baseline and also provides a useful comparison with the interpretable Logistic Regression model.

## Step 5 — Tune Gradient Boosting with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

gb_pipe = Pipeline([
    ('pre', pre),
    ('clf', GradientBoostingClassifier(random_state=RNG))
])

param_grid = {
    'clf__n_estimators': [150, 300],
    'clf__learning_rate': [0.05, 0.1],
    'clf__max_depth': [2, 3]
}

grid = GridSearchCV(
    gb_pipe,
    param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print('best params:', grid.best_params_)
print('best CV ROC-AUC:', round(grid.best_score_, 3))

best_model = grid.best_estimator_

val_proba = best_model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_proba)

print('validation ROC-AUC:', round(val_auc, 3))

## Step 6A — Choose the Operating Threshold on Validation Data

The test set must remain untouched until final evaluation. Therefore, inspect the precision/recall trade-off on the **validation set**, choose an operating threshold, and then freeze that threshold.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.50, 0.40, 0.35, 0.30, 0.25]

print(f"{'thr':>5} {'precision':>10} {'recall':>8} {'f1':>8}")

for threshold in thresholds:
    prediction = (val_proba >= threshold).astype(int)

    print(
        f'{threshold:5.2f} '
        f'{precision_score(y_val, prediction):10.3f} '
        f'{recall_score(y_val, prediction):8.3f} '
        f'{f1_score(y_val, prediction):8.3f}'
    )

### Operating threshold

For this churn scenario we use `0.30` as the operating threshold because missing churners is considered more expensive than sending some unnecessary retention offers. This choice is made before final test evaluation.

In [ ]:
OPERATING_THRESHOLD = 0.30
print('Frozen operating threshold:', OPERATING_THRESHOLD)

## Step 6B — Final Test Evaluation

Now touch the held-out test set once for the final performance estimate.

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

proba = best_model.predict_proba(X_test)[:, 1]
pred_050 = (proba >= 0.50).astype(int)

print('TEST ROC-AUC:', round(roc_auc_score(y_test, proba), 3), '\n')

print(
    classification_report(
        y_test,
        pred_050,
        target_names=['stay', 'churn'],
        digits=3
    )
)

ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred_050),
    display_labels=['stay', 'churn']
).plot(values_format='d')

plt.title('Confusion matrix @ threshold 0.50')
plt.show()

### Final business operating point: threshold 0.30

In [ ]:
pred_operating = (proba >= OPERATING_THRESHOLD).astype(int)

operating_precision = precision_score(y_test, pred_operating)
operating_recall = recall_score(y_test, pred_operating)
operating_f1 = f1_score(y_test, pred_operating)

print('threshold:', OPERATING_THRESHOLD)
print('precision:', round(operating_precision, 3))
print('recall   :', round(operating_recall, 3))
print('f1       :', round(operating_f1, 3))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred_operating),
    display_labels=['stay', 'churn']
).plot(values_format='d')

plt.title('Confusion matrix @ threshold 0.30')
plt.show()

### Business interpretation

- **False negative:** a customer churns but RetailMind fails to flag them.
- **False positive:** a customer would stay but receives a retention intervention.

If losing a subscriber costs substantially more than offering a retention coupon, accepting additional false positives can be rational in exchange for higher recall.

## Step 7 — Interpret the Model

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model,
    X_val,
    y_val,
    n_repeats=8,
    random_state=RNG,
    scoring='roc_auc'
)

imp = pd.Series(
    perm.importances_mean,
    index=X.columns
).sort_values(ascending=False)

print('Permutation importance (ROC-AUC drop when shuffled):')
print(imp.round(4))

### Signed Drivers from Logistic Regression

In [ ]:
logreg = Pipeline([
    ('pre', pre),
    ('clf', LogisticRegression(
        max_iter=2000,
        class_weight='balanced'
    ))
])

logreg.fit(X_train, y_train)

coef = pd.Series(
    logreg.named_steps['clf'].coef_[0],
    index=logreg.named_steps['pre'].get_feature_names_out()
)

print('Most churn-DRIVING features:')
print(coef.sort_values().tail(6).round(2), '\n')

print('Most PROTECTIVE features:')
print(coef.sort_values().head(6).round(2))

### Plot Permutation Importance

In [ ]:
imp.sort_values().plot.barh(
    figsize=(7, 4),
    title='Permutation importance'
)

plt.xlabel('Mean ROC-AUC decrease after shuffling')
plt.tight_layout()
plt.show()

## Step 8 — Manager Recommendation

**Replace this example with your actual values after running the notebook.**

We can identify customers at high risk of churn using account and plan information available before the customer leaves. For retention operations, I recommend using the tuned Gradient Boosting model with the 0.30 threshold selected on validation data, because the business priority is catching more potential churners even if this creates additional retention contacts. The strongest model drivers should be used to prioritize interventions, particularly customers with risky contract/account patterns and repeated support friction. Retention teams should test targeted support and contract-conversion offers, then measure incremental retention and offer cost through a controlled experiment rather than assuming every model alert should receive the same incentive.

## Reflection

### 1. Leakage
Another realistic leakage feature could be `account_closed_date` or a cancellation-reason field. Those fields may only be populated after a customer has already decided to leave. I would investigate suspiciously predictive features by checking when they are generated in the business process, comparing their timestamps with the prediction cutoff, and removing anything that would not exist at scoring time.

### 2. Metric choice
Precision becomes more important when the retention intervention is expensive. For example, if every positive prediction triggers a costly ₹5,000 service credit or an expensive human retention call, too many false positives could make the campaign unprofitable, so the operating threshold should be raised until precision meets the business cost requirement.

### 3. Trust and deployment choice
Because the four models have very similar ROC-AUC, I would compare Gradient Boosting with Logistic Regression rather than automatically choosing the most complex algorithm. If Logistic Regression remains within a few thousandths of ROC-AUC while being easier to explain and maintain, it may be the better production model. The narrow performance spread also suggests that future gains are more likely to come from better behavioural features, product-usage history, billing events, and support interactions than from simply adding increasingly complex algorithms.

### 4. Honesty and test-set isolation
If I repeatedly tried thresholds on the test set and selected whichever produced the best recall or F1, the test set would become part of model development. Reporting performance on that same set would then be optimistically biased because the threshold had indirectly been tuned to those customers. Threshold selection therefore belongs on validation data, while the test set should be reserved for the final unbiased estimate.

## Final Scorecard

In [ ]:
summary = {
    'primary_metric': 'recall on churn (ROC-AUC for selection)',
    'chosen_model': 'GradientBoosting (tuned)',
    'test_roc_auc': round(roc_auc_score(y_test, proba), 3),
    'operating_threshold': OPERATING_THRESHOLD,
    'recall_at_0.30': round(operating_recall, 3),
    'precision_at_0.30': round(operating_precision, 3),
    'top_drivers': imp.head(3).index.tolist()
}

for key, value in summary.items():
    print(f'{key:22s}: {value}')

## Final Validation

In [ ]:
assert df.shape == (4000, 13)
assert 'exit_survey_sent' not in X.columns
assert X_train.shape == (2800, 11)
assert X_val.shape == (600, 11)
assert X_test.shape == (600, 11)
assert Z.shape == (2800, 19)
assert len(results) == 4
assert 0 <= operating_precision <= 1
assert 0 <= operating_recall <= 1
assert 0 <= roc_auc_score(y_test, proba) <= 1

print('All Day 16 capstone validation checks passed.')

# Submission Checklist

- [ ] Restart kernel and run all cells.
- [ ] Confirm dataset shape is `(4000, 13)`.
- [ ] Confirm `exit_survey_sent` is detected and removed.
- [ ] Confirm train / validation / test sizes are 2800 / 600 / 600.
- [ ] Confirm preprocessing creates 19 features.
- [ ] Confirm all four models complete 5-fold CV.
- [ ] Confirm GridSearchCV completes.
- [ ] Confirm threshold is selected using validation data.
- [ ] Confirm final test metrics and confusion matrices render.
- [ ] Confirm permutation importance renders.
- [ ] Replace recommendation numbers with your actual results.
- [ ] Review and personalize the reflection answers.
- [ ] Save as `day16_capstone.ipynb` with outputs visible.